## Example 1: catalog-safe table and column names

In [ ]:
import awswrangler as wr

# Data catalog table and column names allow lowercase letters, digits and underscores. awswrangler lowercases, strips
# accents and turns every other character into `_`. All of this runs locally; no AWS call is made.
cases = {
    "MyNewTable": "mynewtable",
    "Camel Case Name": "camel_case_name",
    "my-table.v2": "my_table_v2",
    "café Ünits": "cafe_units",
    "already_snake": "already_snake",
}
for raw, expected in cases.items():
    assert wr.catalog.sanitize_table_name(raw) == expected, (raw, wr.catalog.sanitize_table_name(raw))
    assert wr.catalog.sanitize_column_name(raw) == expected, (raw, wr.catalog.sanitize_column_name(raw))
print(f"✓ awswrangler {wr.__version__}: " + ", ".join(f"{k!r} -> {v!r}" for k, v in cases.items()))

## Example 2: sanitizing DataFrame column names

In [ ]:
import pandas as pd

frame = pd.DataFrame({"Order Id": [1, 2], "Customer-Name": ["a", "b"], "Unit.Price": [9.5, 3.0]})
original = list(frame.columns)  # the call renames the columns of the frame it is given
clean = wr.catalog.sanitize_dataframe_columns_names(frame)
assert list(clean.columns) == ["order_id", "customer_name", "unit_price"], list(clean.columns)
assert clean["order_id"].tolist() == [1, 2], "the data must be untouched"

# "Total" and "total" collide once lowercased; handle_duplicate_columns="rename" keeps both.
colliding = pd.DataFrame([[1, 2, 3]], columns=["Total", "total", "Count"])
colliding_original = list(colliding.columns)
renamed = wr.catalog.sanitize_dataframe_columns_names(colliding, handle_duplicate_columns="rename")
assert list(renamed.columns) == ["total", "total_1", "count"], list(renamed.columns)
assert renamed.iloc[0].tolist() == [1, 2, 3]
print(f"✓ {original} -> {list(clean.columns)}; {colliding_original} -> {list(renamed.columns)}")

## Example 3: pandas dtypes to SQL column types with `extract_athena_types`

In [ ]:
import datetime
import decimal

frame = pd.DataFrame({
    "id": pd.Series([1, 2], dtype="int64"),
    "small": pd.Series([1, 2], dtype="int32"),
    "price": [1.5, 2.5],
    "name": ["x", "y"],
    "flag": [True, False],
    "ts": pd.to_datetime(["2024-01-01 10:00", "2024-01-02 11:30"]),
    "day": [datetime.date(2024, 1, 1), datetime.date(2024, 1, 2)],
    "amount": [decimal.Decimal("1.23"), decimal.Decimal("4.56")],
    "tags": [["a"], ["b", "c"]],
    "year": ["2024", "2024"],
})
columns, partitions = wr.catalog.extract_athena_types(frame, partition_cols=["year"])
assert columns == {
    "id": "bigint", "small": "int", "price": "double", "name": "string", "flag": "boolean",
    "ts": "timestamp", "day": "date", "amount": "decimal(3,2)", "tags": "array<string>",
}, columns
assert partitions == {"year": "string"}, partitions

# An explicit dtype overrides the inferred one.
columns, _ = wr.catalog.extract_athena_types(frame, partition_cols=["year"], dtype={"amount": "decimal(10,2)"})
assert columns["amount"] == "decimal(10,2)", columns["amount"]
print(f"✓ column types: {columns}; partitions: {partitions}")

## Example 4: global configuration through wr.config and WR_* environment variables

In [ ]:
import os

# Set in code, validated by type, and cleared again by reset().
wr.config.max_cache_seconds = 900
assert wr.config.max_cache_seconds == 900
settings = wr.config.to_pandas()
assert "WR_MAX_CACHE_SECONDS" in set(settings["Env.Variable"]), "every setting has a WR_* variable"

try:
    wr.config.max_cache_seconds = "fifteen minutes"
except wr.exceptions.InvalidConfiguration as error:
    print(f"✓ wrong type rejected: {error}")
else:
    raise AssertionError("wr.config accepted a str for an int setting")

wr.config.reset("max_cache_seconds")
try:
    wr.config.max_cache_seconds
except AttributeError:
    pass
else:
    raise AssertionError("reset() did not clear max_cache_seconds")

# The same setting from the environment: reset() re-reads WR_* variables and converts the type.
os.environ["WR_MAX_CACHE_SECONDS"] = "120"
try:
    wr.config.reset("max_cache_seconds")
    assert wr.config.max_cache_seconds == 120, repr(wr.config.max_cache_seconds)
finally:
    del os.environ["WR_MAX_CACHE_SECONDS"]
    wr.config.reset("max_cache_seconds")
print("✓ wr.config: set, type-checked, reset, and read from WR_MAX_CACHE_SECONDS")